# 🚀 Chapter 08-01b: Build a Supervisor Multi-Agent System

---

## 📖 The Mission

Build a content creation team with:
- 🧑‍💼 **Supervisor** — Coordinates the workflow
- 🔍 **Researcher** — Gathers information
- ✍️ **Writer** — Drafts content
- 🎨 **Editor** — Polishes and finalizes

The supervisor decides who works next!

---

## 🧪 Step 1: Setup

In [ ]:
# !pip install langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ Setup complete!")

## 🧪 Step 2: Define State

In [ ]:
class TeamState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str
    task: str
    research: str
    draft: str
    final_content: str
    iteration: int

WORKERS = ["researcher", "writer", "editor"]
print(f"✅ State defined. Workers: {WORKERS}")

## 🧪 Step 3: Define Agent Nodes

In [ ]:
def supervisor(state: TeamState) -> dict:
    """Supervisor decides which worker to call next."""
    response = llm.invoke([
        SystemMessage(content=f"""You are a team supervisor managing a content creation team.
Workers available: {', '.join(WORKERS)}
Workflow: researcher first, then writer, then editor.
If all work is done, respond with FINISH.
Respond with ONLY the next worker name or FINISH."""),
        *state["messages"]
    ])
    next_agent = response.content.strip().lower()
    print(f"  🧑‍💼 Supervisor: Next → {next_agent}")
    return {
        "next_agent": next_agent,
        "messages": [AIMessage(content=f"Supervisor delegates to: {next_agent}")],
        "iteration": state.get("iteration", 0) + 1
    }

def researcher_agent(state: TeamState) -> dict:
    """Researches the topic."""
    response = llm.invoke([
        SystemMessage(content="You are an expert researcher. Provide key facts, data points, and interesting angles about the topic. Be concise but thorough."),
        HumanMessage(content=f"Research this: {state['task']}")
    ])
    print(f"  🔍 Researcher completed work")
    return {
        "research": response.content,
        "messages": [AIMessage(content=f"Research complete: {response.content[:100]}...")]
    }

def writer_agent(state: TeamState) -> dict:
    """Writes content based on research."""
    response = llm.invoke([
        SystemMessage(content="You are a skilled content writer. Write engaging, clear content based on the research provided. Include an intro, body, and conclusion."),
        HumanMessage(content=f"Task: {state['task']}\nResearch: {state.get('research', 'N/A')}\nWrite the content.")
    ])
    print(f"  ✍️ Writer completed draft")
    return {
        "draft": response.content,
        "messages": [AIMessage(content=f"Draft written: {response.content[:100]}...")]
    }

def editor_agent(state: TeamState) -> dict:
    """Edits and polishes the content."""
    response = llm.invoke([
        SystemMessage(content="You are a professional editor. Polish the content: fix grammar, improve flow, make it engaging. Return the final polished version."),
        HumanMessage(content=f"Edit this content:\n{state.get('draft', 'N/A')}")
    ])
    print(f"  🎨 Editor polished content")
    return {
        "final_content": response.content,
        "messages": [AIMessage(content=f"Editing complete. Final content ready.")]
    }

print("✅ All agent nodes defined!")

## 🧪 Step 4: Build the Graph

In [ ]:
def route_supervisor(state: TeamState) -> str:
    next_agent = state.get("next_agent", "finish").lower()
    if next_agent == "finish" or state.get("iteration", 0) > 6:
        return "end"
    if next_agent in WORKERS:
        return next_agent
    return "end"  # Safety fallback

# Build graph
graph = StateGraph(TeamState)

graph.add_node("supervisor", supervisor)
graph.add_node("researcher", researcher_agent)
graph.add_node("writer", writer_agent)
graph.add_node("editor", editor_agent)

graph.set_entry_point("supervisor")

# Supervisor routes to workers or END
graph.add_conditional_edges("supervisor", route_supervisor, {
    "researcher": "researcher",
    "writer": "writer",
    "editor": "editor",
    "end": END
})

# All workers go back to supervisor
for worker in WORKERS:
    graph.add_edge(worker, "supervisor")

# Compile
app = graph.compile(checkpointer=MemorySaver())
print("✅ Multi-agent graph compiled!")
try:
    print(app.get_graph().draw_ascii())
except Exception:
    print("Graph: supervisor → [researcher|writer|editor] → supervisor → END")

## 🧪 Step 5: Run the Team!

In [ ]:
config = {"configurable": {"thread_id": "content-team-1"}}

print("🚀 Content Creation Team Starting!")
print("=" * 60)

result = app.invoke({
    "messages": [HumanMessage(content="Create a blog post about AI agents in 2024")],
    "task": "Write a compelling blog post about how AI agents are transforming software development in 2024",
    "next_agent": "",
    "research": "",
    "draft": "",
    "final_content": "",
    "iteration": 0
}, config)

print("\n" + "=" * 60)
print(f"📊 Total iterations: {result['iteration']}")

In [ ]:
# Print final content
print("📄 FINAL BLOG POST")
print("=" * 60)
content = result.get("final_content") or result.get("draft", "No content generated")
print(content)

## ✍️ Practice Challenge

Add a **"seo_optimizer"** worker that takes the final content and adds SEO-friendly title, meta description, and keywords. Update the supervisor to call it after the editor.

<details>
<summary>💡 Hint</summary>

1. Add `"seo_optimizer"` to WORKERS list
2. Create `seo_agent()` function with SEO-focused system prompt
3. Add node: `graph.add_node("seo_optimizer", seo_agent)`
4. Add edge back to supervisor: `graph.add_edge("seo_optimizer", "supervisor")`

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Supervisor agent coordinates worker agents via routing.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Workers → supervisor cycle creates dynamic task delegation.    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Max iterations prevent infinite loops.                         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Shared state lets workers build on each other's output.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 This is the standard production multi-agent pattern.           │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: CrewAI, AutoGen & Framework Comparison!

---

### 🎉 Your First Multi-Agent Team!
Supervisor → Workers → Supervisor — coordinated AI teamwork. 🚀